#### Dependencies

In [1]:
import os
from azure.core.exceptions import HttpResponseError
from azure.ai.ml import MLClient, command, Input, Output, dsl
from azure.ai.ml.entities import AmlCompute, Environment, CommandJob, BuildContext, Data
from azure.identity import DefaultAzureCredential, ManagedIdentityCredential
from azureml.core import Workspace, Experiment, Datastore
from azure.ai.ml.constants import AssetTypes

#### Create Handle

In [2]:
# obtain workspace auth details from local config file
ws = Workspace.from_config()

# choose credential access route
credential = ManagedIdentityCredential()
#credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=ws.subscription_id,
    resource_group_name=ws.resource_group,
    workspace_name=ws.name
)

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


#### Create YAML file

In [3]:
# create env folder if one doesn't exist
dependencies_dir = "./env"
os.makedirs(dependencies_dir, exist_ok=True)

In [8]:
%%writefile {dependencies_dir}/environment.yml
name: ray-env-chatgpt
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
    - torch
    - torchvision
    - ray[default]

Overwriting ./env/environment.yml


#### Define Environment

In [9]:
# Define environment
env = Environment(
    name="ray-env-chatgpt",
    description="Ray training environment to experiment running ray on AzureML",
    tags={"ray": "ray-torch"},
    conda_file=os.path.join(dependencies_dir, "environment.yml"),
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04"
)

ml_client.environments.create_or_update(env)

print(
    f"Environment with name {env} is registered to workspace, the env version is {env.version}")

Environment with name conda_file:
  channels:
  - conda-forge
  dependencies:
  - python=3.10
  - pip
  - pip:
    - torch
    - torchvision
    - ray[default]
  name: ray-env-chatgpt
description: Ray training environment to experiment running ray on AzureML
image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04
name: ray-env-chatgpt
tags:
  ray: ray-torch
version: '1'
 is registered to workspace, the env version is 1


##### Define and Create Data Asset

In [4]:
data_path = "./data/train.pt"

# Define the Data Asset Object
my_data = Data(
    path = data_path,
    type=AssetTypes.URI_FILE,
    description="This data set represents the fashionMNIST train set images",
    name="fashion_mnist_train_data",
    version="1"
)

ml_client.data.create_or_update(my_data)

NameError: name 'ml' is not defined

#### Define Job

In [3]:
compute_name = 'mlc-kamunya3'
#compute_name = 'kamunya-vm-1'

# Define command job
job = command(
    code="./src",
    command="python train.py --data_dir ${{inputs.fashion_mnist}}",
    inputs={
        "fashion_mnist": Input(
            type="uri_folder",
            path="./data",
            mode="mount"
        )
    },

    environment="ray-env-chatgpt@latest",
    compute=compute_name,
    experiment_name="fashion-mnist-ray-chatgpt",
    shm_size = "10g"
)

returned_job = ml_client.jobs.create_or_update(job)
print(returned_job.studio_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.0 MBs): 100%|████████

https://ml.azure.com/runs/keen_kitten_7wzw9cv9tp?wsid=/subscriptions/6edfe71f-db9c-4622-8259-95e374d29cf2/resourcegroups/kam-east/workspaces/kamunya-ml-first&tid=89660ab5-e50f-4bea-983d-53e5468b6aba


### References

1. Launching Ray Clusters on Azure - https://docs.ray.io/en/latest/cluster/vms/user-guides/launching-clusters/azure.html$0
2. PyTorch 2D Convolution - https://www.youtube.com/watch?v=n8Mey4o8gLc&t=780s$0